# Pemodelan — LSTM Kuantil

## 1. Konfigurasi

In [1]:
# Notebook ini menjalankan LSTM kuantil dari awal sampai akhir: benchmark cpu/cuda (§5),
# anggaran pencarian (§6), pencarian hyperparameter (§7), walk-forward lima fold (§8),
# pengulangan tiga seed (§9), model final (§10), ringkasan hasil (§11), dan perbandingan tiga
# arah dengan Random Forest & XGBoost (§12). Desember 2025 tidak dibuka di notebook ini.
#
# Yang diinline di sini: kode modelnya (§3-§4, salinan verbatim dari
# utils/modelling/model_lstm.py) dan jendela sekuensnya (§2, dari
# utils/modelling/sequence_windows.py) — keduanya khusus LSTM dan tidak dipakai notebook lain.
# §13 membandingkan keduanya dengan sumbernya di utils. Mesin bersama tetap diimpor dari utils:
# modeling_prep, walk_forward, evaluation, model_common, purging, run_config (keputusan pemilik
# proyek 2026-08-26).
#
# Desain: docs/superpowers/specs/2026-08-19-lstm-modeling-design.md
# Hasil terukur: belum ada untuk kriteria multi-kuantil. Angka era kuantil-tunggal
# diarsipkan di docs/bak/hasil-modeling-lstm.single-quantile.bak.md dan TIDAK sebanding
# dengan K1 — dokumen Fase 3 ditulis setelah pencarian di §8 selesai.
import sys
import time
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import torch
from numpy.lib.stride_tricks import sliding_window_view
from torch import nn


def find_base_dir(start=None) -> Path:
    """Cari root repo — folder pertama ke atas yang berisi `dataset/csv/`."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "dataset" / "csv").is_dir():
            return candidate
    raise RuntimeError(f"Root repo tidak ditemukan dari {start}")


BASE_DIR = find_base_dir()

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from utils.modelling import evaluation, model_common, modeling_prep, purging, run_config
from utils.modelling import walk_forward

print(f"BASE_DIR = {BASE_DIR}")

BASE_DIR = /Users/ramapdp/Project/Personal/forecast-scm


## 2. Jendela sekuens

In [2]:
# Indeks jendela geser yang mengubah panel harian menjadi sekuens panjang LOOKBACK, plus
# penjagaannya: panel harus padat per segmen dan tiap jendela harus muat utuh di dalam satu
# segmen — jendela yang menyeberangi penutupan cabang akan mencampur dua rezim berbeda.
def _split_columns(feature_cols: list) -> tuple:
    idx_cols = [col for col in feature_cols if col.endswith("_idx")]
    dynamic_cols = [col for col in feature_cols if not col.endswith("_idx")]
    return dynamic_cols, idx_cols


def build_index(
    panel: pd.DataFrame,
    feature_cols: Optional[list] = None,
    lookback: int = modeling_prep.LOOKBACK,
    date_col: str = modeling_prep.DATE_COL,
    pair_cols: Optional[list] = None,
) -> dict:
    """One contiguous view of the whole panel, plus everything needed to
    address windows inside it.

    The panel passed here is the **full** frame, not `eligible_rows()`. A
    window for a 1 July validation row reaches back into June, over rows that
    the 28-day warm-up cut and the fold purge both remove — rows that appear
    in neither frame `walk_forward.run_fold()` hands a model. Reading their
    *features* is safe: every window ends at its own prediction row, and every
    lag and rolling feature stops at H-1, so no target value can enter a
    window. Purging protects against training on those rows' labels, which
    still never happens.
    """
    feature_cols = list(feature_cols or modeling_prep.FEATURE_COLS)
    dynamic_cols, idx_cols = _split_columns(feature_cols)

    # G2. A target channel inside a window would be a perfect predictor of
    # itself and would not change a single tensor shape.
    for target_col in (modeling_prep.EVAL_TARGET_COL,
                       modeling_prep.TRAIN_TARGET_COL):
        if target_col in dynamic_cols:
            raise ValueError(
                f"{target_col} tidak boleh menjadi kolom dinamis"
            )

    pair_cols = modeling_prep._resolve_pair_cols(panel, pair_cols)
    frame = panel.sort_values(pair_cols + [date_col]).reset_index(drop=True)

    values = np.ascontiguousarray(frame[dynamic_cols].to_numpy(dtype="float32"))
    cats = np.ascontiguousarray(frame[idx_cols].to_numpy(dtype="int16"))
    dates = frame[date_col].to_numpy("datetime64[D]")

    grouped = frame.groupby(pair_cols, observed=True, sort=False)
    positions = grouped.cumcount().to_numpy()
    segment_code = grouped.ngroup().to_numpy()

    _assert_dense(dates, positions, segment_code)
    _assert_windows_fit(dates, positions, segment_code, lookback)

    key_cols = list(pair_cols) + [date_col]
    lookup = pd.Series(
        np.arange(len(frame), dtype=np.int64),
        index=pd.MultiIndex.from_frame(frame[key_cols]),
    )

    return {
        "values": values,
        "cats": cats,
        "dates": dates,
        "positions": positions,
        "segment_code": segment_code,
        "lookup": lookup,
        "key_cols": key_cols,
        "feature_cols": feature_cols,
        "dynamic_cols": dynamic_cols,
        "idx_cols": idx_cols,
        "lookback": lookback,
    }


def _assert_dense(dates, positions, segment_code) -> None:
    """G1, first half: consecutive positions are consecutive days."""
    inside = positions > 0
    if not inside.any():
        return
    step = (dates[1:] - dates[:-1]).astype("timedelta64[D]").astype(np.int64)
    same_segment = segment_code[1:] == segment_code[:-1]
    bad = same_segment & (step != 1)
    if bad.any():
        first = int(np.flatnonzero(bad)[0]) + 1
        raise ValueError(
            f"celah tanggal di dalam segmen pada posisi {first} "
            f"({dates[first - 1]} -> {dates[first]}); "
            "aritmetika posisi tidak lagi sama dengan aritmetika tanggal"
        )


def _assert_windows_fit(dates, positions, segment_code, lookback) -> None:
    """G1 second half and G6: every usable window stays inside one segment,
    spans exactly `lookback` days, and never reaches past its own row.
    """
    ends = np.flatnonzero(positions >= lookback)
    if ends.size == 0:
        return
    starts = ends - lookback + 1
    if not np.array_equal(segment_code[starts], segment_code[ends]):
        raise ValueError("ada window yang melintasi batas segmen")
    span = (dates[ends] - dates[starts]).astype("timedelta64[D]").astype(np.int64)
    if not np.all(span == lookback - 1):
        raise ValueError(
            f"ada window yang tidak mencakup tepat {lookback} hari berurutan"
        )


def window_ends(index: dict, frame: pd.DataFrame) -> np.ndarray:
    """Row positions in `index["values"]` for a frame of prediction rows.

    Returned in the frame's **own row order**, so a caller can line
    predictions up against `valid.index` without a join — which is what
    `walk_forward.run_fold()` assumes when it wraps the array in a Series.
    """
    key = pd.MultiIndex.from_frame(frame[index["key_cols"]])
    ends = index["lookup"].reindex(key).to_numpy()
    missing = pd.isna(ends)
    if missing.any():
        raise ValueError(
            f"{int(missing.sum())} baris tidak ditemukan di panel; "
            "frame prediksi harus berasal dari panel yang sama dengan indeks"
        )
    return ends.astype(np.int64)


def gather(
    values: np.ndarray,
    ends: np.ndarray,
    lookback: int = modeling_prep.LOOKBACK,
) -> np.ndarray:
    """`(len(ends), lookback, n_features)` — the window ending at each position.

    `values` is taken as an argument rather than read from the index so a
    per-fold scaled copy can be passed without rebuilding anything.

    `sliding_window_view` costs nothing: it is a strided view over `values`.
    The only allocation is the batch itself, produced by the fancy index.
    """
    if len(ends) == 0:
        return np.empty((0, lookback, values.shape[1]), dtype="float32")
    if ends.min() < lookback - 1:
        raise ValueError(
            f"posisi akhir {int(ends.min())} terlalu awal untuk window "
            f"{lookback} hari"
        )
    windows = sliding_window_view(values, lookback, axis=0)
    return np.ascontiguousarray(
        windows[ends - lookback + 1].transpose(0, 2, 1)
    )

## 3. Arsitektur & anggaran

In [3]:
QUANTILE = 0.9


QUANTILES = evaluation.QUANTILE_SET_A


ES_TAIL_DAYS = 30


EARLY_STOPPING_EPOCHS = 5


MAX_EPOCHS = 100


BUDGET_SECONDS = 28_800


MIN_CANDIDATES = 6


MAX_CANDIDATES = 20


DEFAULT_PARAMS = {
    "hidden_size": 128,
    "num_layers": 1,
    "dropout": 0.2,
    "learning_rate": 1e-3,
    "batch_size": 1024,
    "log_target": False,
    "grad_clip": 1.0,
    "random_state": 42,
}


SEARCH_SPACE = {
    "hidden_size": [64, 128, 256],
    "num_layers": [1, 2],
    "dropout": [0.0, 0.2, 0.3],
    "learning_rate": [3e-4, 1e-3],
    "batch_size": [1024, 2048],
    "log_target": [False, True],
}


N_CANDIDATES = 30


def pinball_loss(
    prediction: torch.Tensor,
    target: torch.Tensor,
    quantiles: tuple = QUANTILES,
) -> torch.Tensor:
    """The training objective *is* the selection criterion.

    `prediction` is `(batch, len(quantiles))`, `target` is `(batch,)` and
    broadcasts across the grid. The sum runs over quantiles and the mean over
    rows: summing keeps each point's gradient at its own scale, so the extreme
    quantiles are not drowned out by the dense middle of the grid, while the
    mean over rows keeps the number independent of batch size.

    K1 is the *mean* over quantiles, so it and this loss differ by the constant
    `len(quantiles)` and by nothing else — which preserves the property
    `reg:quantileerror` gives XGBoost: a model cannot win the fit and lose the
    metric.
    """
    alphas = torch.as_tensor(quantiles, dtype=prediction.dtype,
                             device=prediction.device).view(1, -1)
    difference = target.view(-1, 1) - prediction
    return torch.maximum(alphas * difference,
                         (alphas - 1.0) * difference).sum(dim=1).mean()


def embedding_sizes(
    mapping: Optional[dict] = None,
    idx_cols: Optional[list] = None,
) -> list:
    """`(num_embeddings, embedding_dim)` per `_idx` column.

    `num_embeddings` comes from `category_mapping.json` — the highest index
    plus one, which already covers the reserved UNKNOWN slot at 0 — and never
    from the values that happen to appear in a fold's training rows. A branch
    opening after this model is trained maps to 0 and must stay in range; the
    alternative fails months later, in production, with an index error.
    """
    mapping = mapping if mapping is not None else modeling_prep.load_category_mapping()
    idx_cols = idx_cols or model_common.IDX_COLS
    sizes = []
    for col in idx_cols:
        source = col[: -len("_idx")]
        num_embeddings = max(mapping[source].values()) + 1
        sizes.append((num_embeddings, min(16, (num_embeddings + 1) // 2)))
    return sizes

In [4]:
class QuantileLSTM(nn.Module):
    """49 dynamic channels through the LSTM, 7 categoricals through embeddings.

    The categoricals are read at the prediction row, not repeated across the
    window: `Kategori Barang_idx` changes inside 301 real segments, so "the
    segment's category" is not a well-defined thing to repeat.
    """

    def __init__(
        self,
        n_dynamic: int,
        sizes: list,
        hidden_size: int = 128,
        num_layers: int = 2,
        dropout: float = 0.2,
        n_quantiles: int = len(QUANTILES),
    ):
        super().__init__()
        self.embeddings = nn.ModuleList(
            [nn.Embedding(count, dim) for count, dim in sizes]
        )
        self.lstm = nn.LSTM(
            input_size=n_dynamic,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            # nn.LSTM ignores this when num_layers == 1, which is why the head
            # below always applies dropout: otherwise the searched flag would
            # be meaningless across half the space.
            dropout=dropout if num_layers > 1 else 0.0,
        )
        width = hidden_size + sum(dim for _, dim in sizes)
        # One output neuron per quantile, on a shared trunk: every point reads
        # the same LSTM state and the same embeddings, which is what makes this
        # cheaper than len(QUANTILE_SET) separate networks and what lets the
        # quantiles inform each other. Nothing here forces the outputs to be
        # monotone in tau — crossing is measured, not designed away.
        self.head = nn.Sequential(
            nn.Linear(width, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, n_quantiles),
        )

    def forward(self, x_dynamic: torch.Tensor, x_cats: torch.Tensor) -> torch.Tensor:
        output, _ = self.lstm(x_dynamic)
        last = output[:, -1, :]
        embedded = [layer(x_cats[:, position])
                    for position, layer in enumerate(self.embeddings)]
        return self.head(torch.cat([last, *embedded], dim=1))

In [5]:
def build_model(params: dict, n_dynamic: int, sizes: list, seed: int,
                n_quantiles: int = len(QUANTILES)) -> QuantileLSTM:
    """Seeded construction, so the two fits of the two-fit protocol start
    from identical weights and `best_epoch` means the same thing in both.
    """
    torch.manual_seed(seed)
    return QuantileLSTM(
        n_dynamic=n_dynamic,
        sizes=sizes,
        hidden_size=params["hidden_size"],
        num_layers=params["num_layers"],
        dropout=params["dropout"],
        n_quantiles=n_quantiles,
    )


def resolve_device(name: str = "cpu") -> torch.device:
    """CPU by default, deliberately.

    MPS has no fused LSTM kernel and at these hidden sizes is often slower
    than CPU, so the benchmark measures both and records which one won rather
    than a default silently choosing.

    CUDA is checked the same way. `torch.device("cuda")` on a machine without
    one constructs happily and only fails later, inside a training loop, far
    from the line that asked for it — so the benchmark cell would pay the
    window index before finding out. Raising here is what lets that cell list
    every device and skip the ones this machine does not have.
    """
    if name == "mps" and not torch.backends.mps.is_available():
        raise ValueError("MPS tidak tersedia di mesin ini")
    if name.startswith("cuda") and not torch.cuda.is_available():
        raise ValueError("CUDA tidak tersedia di mesin ini")
    return torch.device(name)


def candidate_budget(
    sec_per_epoch: float,
    best_epoch: int,
    budget_seconds: int = BUDGET_SECONDS,
    patience: int = EARLY_STOPPING_EPOCHS,
    minimum: int = MIN_CANDIDATES,
    maximum: int = MAX_CANDIDATES,
) -> int:
    """How many candidates fit inside the wall-clock ceiling.

    One two-fit candidate costs about
    `sec_per_epoch * (2 * best_epoch + patience)`, and each is scored on two
    folds. Fold 3's training window is shorter than fold 5's, so charging both
    at fold 5's measured rate is conservative.

    Below `minimum` this raises instead of clamping upward. Clamping up would
    be a silent overrun of the ceiling, and the spec is explicit that a
    too-small N is the signal to shrink the search space — a decision for the
    operator, not for this function.
    """
    per_fit = sec_per_epoch * (2 * best_epoch + patience)
    raw = int(budget_seconds // (2 * per_fit))
    if raw < minimum:
        raise ValueError(
            f"anggaran hanya cukup untuk {raw} kandidat (<{minimum}); "
            f"perkecil ruang search atau turunkan ongkos per fit — "
            f"jangan naikkan plafon {budget_seconds}s diam-diam"
        )
    return min(raw, maximum)


def scale_values(values: np.ndarray, scaler: dict, dynamic_cols: list) -> np.ndarray:
    """Standardise the whole panel matrix with one fold's scaler.

    The scaler is fit on that fold's training rows only. Applying it to every
    row, context rows included, is safe — what leaks is *fitting* it outside
    the training window, never applying it.
    """
    mean = np.array([scaler[col][0] for col in dynamic_cols], dtype="float32")
    std = np.array([scaler[col][1] for col in dynamic_cols], dtype="float32")
    return ((values - mean) / std).astype("float32")

## 4. Pelatihan & prediksi

In [6]:
def _shuffled_batches(count: int, batch_size: int, generator) -> list:
    order = torch.randperm(count, generator=generator).numpy()
    return [order[start:start + batch_size] for start in range(0, count, batch_size)]


def _to_tensors(scaled, cats, ends, lookback, device):
    windows = gather(scaled, ends, lookback=lookback)
    x_dynamic = torch.from_numpy(windows).to(device)
    x_cats = torch.from_numpy(cats[ends].astype("int64")).to(device)
    return x_dynamic, x_cats


def run_epoch(
    model: QuantileLSTM,
    optimizer,
    scaled: np.ndarray,
    cats: np.ndarray,
    ends: np.ndarray,
    targets: np.ndarray,
    params: dict,
    quantiles: tuple,
    generator,
    device,
    lookback: int = modeling_prep.LOOKBACK,
) -> float:
    """One pass over the training windows, returning the mean loss.

    Windows are shuffled across segments. Each one is self-contained, so no
    ordering needs preserving between batches.
    """
    model.train()
    total, seen = 0.0, 0
    for batch in _shuffled_batches(len(ends), params["batch_size"], generator):
        x_dynamic, x_cats = _to_tensors(scaled, cats, ends[batch], lookback, device)
        y = torch.from_numpy(targets[batch].astype("float32")).to(device)

        optimizer.zero_grad()
        loss = pinball_loss(model(x_dynamic, x_cats), y, quantiles)
        if not torch.isfinite(loss):
            # Fails this candidate through run_search's existing catch tuple.
            # RuntimeError is not raised here on purpose: PyTorch uses it for
            # genuine bugs as well as OOM, so widening the tuple would launder
            # bugs into NaN rows.
            raise ValueError(
                "loss LSTM menjadi NaN/inf — kandidat digagalkan"
            )
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), params["grad_clip"])
        optimizer.step()

        total += float(loss.detach()) * len(batch)
        seen += len(batch)
    return total / max(seen, 1)


@torch.no_grad()
def predict(
    model: QuantileLSTM,
    scaled: np.ndarray,
    cats: np.ndarray,
    ends: np.ndarray,
    device,
    lookback: int = modeling_prep.LOOKBACK,
    batch_size: int = 4096,
) -> np.ndarray:
    """Predictions in `ends` order, one column per quantile point.

    Returned exactly as the head produced them — no sort. A post-hoc sort
    would drive `evaluation.crossing_rate()` to zero without making the
    distribution any more coherent, which is hiding the measurement rather
    than taking it.
    """
    model.eval()
    if len(ends) == 0:
        return np.empty((0, model.head[-1].out_features), dtype="float32")
    parts = []
    for start in range(0, len(ends), batch_size):
        chunk = ends[start:start + batch_size]
        x_dynamic, x_cats = _to_tensors(scaled, cats, chunk, lookback, device)
        parts.append(model(x_dynamic, x_cats).cpu().numpy())
    return np.concatenate(parts)


def _evaluate(model, scaled, cats, ends, targets, quantiles, device, lookback):
    """The early-stopping metric: mean pinball across the grid — K1's own
    definition, so the epoch that wins here is the epoch that wins the
    criterion the candidate is later ranked on.

    The mean rather than the training loss's sum. They order epochs
    identically (they differ by len(quantiles)), and reading the same number
    the results tables carry is worth more than saving a multiplication.
    """
    prediction = predict(model, scaled, cats, ends, device=device, lookback=lookback)
    alphas = np.asarray(quantiles, dtype="float64").reshape(1, -1)
    difference = (targets.astype("float64").reshape(-1, 1)
                  - prediction.astype("float64"))
    return float(np.maximum(alphas * difference,
                            (alphas - 1.0) * difference).mean())


def fit_with_early_stopping(
    params: dict,
    index: dict,
    fit_ends: np.ndarray,
    fit_targets: np.ndarray,
    es_ends: np.ndarray,
    es_targets: np.ndarray,
    quantiles: tuple,
    sizes: list,
    device,
    scaled: Optional[np.ndarray] = None,
    max_epochs: int = MAX_EPOCHS,
    patience: int = EARLY_STOPPING_EPOCHS,
    lookback: int = modeling_prep.LOOKBACK,
) -> tuple:
    """Fit on the purged rows, stop on the tail, report the epoch that won.

    Under `log_target` the stopping metric is computed on the log scale. That
    is sound: early stopping only chooses an epoch count *within* one
    candidate. Candidates are compared to each other by pinball on the
    original scale, after inversion.
    """
    scaled = index["values"] if scaled is None else scaled
    model = build_model(params, len(index["dynamic_cols"]), sizes,
                        params["random_state"], n_quantiles=len(quantiles))
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])
    generator = torch.Generator().manual_seed(params["random_state"])

    best_score, best_epoch, since_improvement = float("inf"), 1, 0
    for epoch in range(1, max_epochs + 1):
        run_epoch(model, optimizer, scaled, index["cats"], fit_ends, fit_targets,
                  params, quantiles, generator, device, lookback)
        score = _evaluate(model, scaled, index["cats"], es_ends, es_targets,
                          quantiles, device, lookback)
        if score < best_score:
            best_score, best_epoch, since_improvement = score, epoch, 0
        else:
            since_improvement += 1
            if since_improvement >= patience:
                break
    model.best_score = best_score
    return model, best_epoch


def fit_epochs(
    params: dict,
    index: dict,
    ends: np.ndarray,
    targets: np.ndarray,
    epochs: int,
    quantiles: tuple,
    sizes: list,
    device,
    scaled: Optional[np.ndarray] = None,
    lookback: int = modeling_prep.LOOKBACK,
) -> QuantileLSTM:
    """The second fit: same seed, all training rows, a fixed epoch count.

    One epoch here contains about 5% more gradient steps than one epoch of the
    first fit, because the early-stopping tail is back in. That is accepted:
    pinning by epoch means "the same number of passes over the data", which is
    the more meaningful invariant than a fixed step count.
    """
    scaled = index["values"] if scaled is None else scaled
    model = build_model(params, len(index["dynamic_cols"]), sizes,
                        params["random_state"], n_quantiles=len(quantiles))
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])
    generator = torch.Generator().manual_seed(params["random_state"])

    for _ in range(epochs):
        run_epoch(model, optimizer, scaled, index["cats"], ends, targets,
                  params, quantiles, generator, device, lookback)
    model.epochs_run = epochs
    return model

In [7]:
def _target(frame, params: dict) -> np.ndarray:
    values = model_common.train_target(frame)
    return (np.log1p(values) if params["log_target"] else values).astype("float32")


def _assert_no_december(index: dict, ends: np.ndarray,
                        test_start=modeling_prep.TEST_START) -> None:
    """G5. Redundant with the fold definitions, and kept anyway: the cost of
    one accidental leak is the credibility of the final number.
    """
    if len(ends) and index["dates"][ends].max() >= np.datetime64(test_start, "D"):
        raise ValueError("ada window yang menyentuh Desember 2025")


def _assert_train_precedes_valid(index: dict, train_ends: np.ndarray,
                                 valid_ends: np.ndarray) -> None:
    """G3. Checks the *position mapping*, not `fold_train_mask` — the frames
    were already split correctly by `walk_forward`, so what this can still
    catch is `window_ends()` handing back the wrong rows.
    """
    if len(train_ends) and len(valid_ends):
        if index["dates"][train_ends].max() >= index["dates"][valid_ends].min():
            raise ValueError(
                "posisi window training tidak seluruhnya mendahului validasi"
            )


def make_fit_predict(
    params: Optional[dict] = None,
    index: Optional[dict] = None,
    feature_cols: Optional[list] = None,
    quantiles: tuple = QUANTILES,
    tail_days: int = ES_TAIL_DAYS,
    max_epochs: int = MAX_EPOCHS,
    patience: int = EARLY_STOPPING_EPOCHS,
    device_name: str = "cpu",
    sizes: Optional[list] = None,
) -> "object":
    """The callable `walk_forward.run_fold()` injects.

    Two fits. The first runs on the purged fit rows with the 30-day tail as
    its eval set and reports the epoch that won. The second discards that
    model, re-initialises from the same seed, and trains on **every** training
    row for exactly that many epochs — so the model producing the reported
    predictions has seen the same population the Random Forest and XGBoost
    were trained on.

    Best epochs are recorded on the returned callable rather than returned,
    because `walk_forward` accepts predictions and nothing else — and their
    spread across folds is worth reporting.
    """
    if index is None:
        raise ValueError("make_fit_predict butuh indeks dari bind_panel()")
    params = {**DEFAULT_PARAMS, **(params or {})}
    quantiles = tuple(quantiles)
    device = resolve_device(device_name)
    # From category_mapping.json unless the caller supplies its own — tests do,
    # because a synthetic _idx fixture column has no entry in that file.
    sizes = sizes if sizes is not None else embedding_sizes(
        idx_cols=index["idx_cols"])
    lookback = index["lookback"]

    def fit_predict(train, valid) -> np.ndarray:
        model_common.assert_no_nan(train, index["feature_cols"])
        model_common.assert_no_nan(valid, index["feature_cols"])

        train_ends = window_ends(index, train)
        valid_ends = window_ends(index, valid)
        _assert_train_precedes_valid(index, train_ends, valid_ends)
        _assert_no_december(index, np.concatenate([train_ends, valid_ends]))

        # One scaler for the whole fold, used by both fits. The tail's
        # statistics sit inside the training window so nothing leaks into
        # validation; sharing it is what makes best_epoch transfer between
        # two fits that would otherwise see differently scaled inputs.
        scaler = modeling_prep.fit_scaler(train, index["dynamic_cols"])
        scaled = scale_values(index["values"], scaler, index["dynamic_cols"])

        fit_rows, es_rows = model_common.split_early_stopping(
            train, tail_days=tail_days)
        fit_ends = window_ends(index, fit_rows)
        es_ends = window_ends(index, es_rows)

        _, best_epoch = fit_with_early_stopping(
            params, index, fit_ends, _target(fit_rows, params),
            es_ends, _target(es_rows, params), quantiles=quantiles, sizes=sizes,
            device=device, scaled=scaled, max_epochs=max_epochs,
            patience=patience, lookback=lookback)
        fit_predict.best_epochs.append(int(best_epoch))

        model = fit_epochs(
            params, index, train_ends, _target(train, params), epochs=best_epoch,
            quantiles=quantiles, sizes=sizes, device=device, scaled=scaled,
            lookback=lookback)

        prediction = predict(model, scaled, index["cats"], valid_ends,
                             device=device, lookback=lookback)
        prediction = np.asarray(prediction, dtype="float64").reshape(
            len(valid_ends), len(quantiles))
        if params["log_target"]:
            prediction = modeling_prep.inverse_log_target(prediction)
        # A negative shipment quantity is not a thing.
        return np.clip(prediction, 0.0, None)

    fit_predict.best_epochs = []
    fit_predict.index = index
    return fit_predict


def bind_panel(
    panel,
    feature_cols: Optional[list] = None,
    lookback: int = modeling_prep.LOOKBACK,
    device_name: str = "cpu",
    tail_days: int = ES_TAIL_DAYS,
    max_epochs: int = MAX_EPOCHS,
    patience: int = EARLY_STOPPING_EPOCHS,
    sizes: Optional[list] = None,
):
    """Give `model_common.run_search()` a callable of the signature it expects.

    `run_search` calls `make_fit_predict(candidate, feature_cols=...,
    quantiles=...)`. There is no slot for the panel, and adding one would
    change a signature the other two models already satisfy — so the panel is
    bound here instead.

    The window index is built **once**. It costs a sort of 1.5M rows;
    rebuilding it per candidate would repeat that N x 2 times for nothing.
    Which is also why the device is resolved *before* the index rather than at
    the first fit: a device this machine does not have should cost nothing.
    """
    resolve_device(device_name)
    index = build_index(panel, feature_cols=feature_cols,
                                         lookback=lookback)

    def make(params=None, feature_cols=None, quantiles: tuple = QUANTILES):
        if feature_cols is not None and list(feature_cols) != index["feature_cols"]:
            raise ValueError(
                "feature_cols berbeda dari yang dipakai membangun indeks"
            )
        return make_fit_predict(params, index=index, quantiles=quantiles,
                                tail_days=tail_days, max_epochs=max_epochs,
                                patience=patience, device_name=device_name,
                                sizes=sizes)

    make.index = index
    return make

In [8]:
MODEL_FILE = str(BASE_DIR / "models/lstm_q90.joblib")


BEST_PARAMS_FILE = str(BASE_DIR / "dataset/model_ready/lstm_best_params.json")


SEED_REPEATS_FILE = str(BASE_DIR / "dataset/model_ready/lstm_seed_repeats.csv")


SEED_REPEATS = (42, 43, 44)


def fit_final(
    df,
    params: dict,
    feature_cols: Optional[list] = None,
    lookback: int = modeling_prep.LOOKBACK,
    tail_days: int = ES_TAIL_DAYS,
    max_epochs: int = MAX_EPOCHS,
    patience: int = EARLY_STOPPING_EPOCHS,
    quantiles: tuple = QUANTILES,
    device_name: str = "cpu",
    sizes: Optional[list] = None,
    date_col: str = modeling_prep.DATE_COL,
    test_start=modeling_prep.TEST_START,
) -> dict:
    """Fit on every eligible row before December, purged at that boundary.

    Eligibility comes from `walk_forward.eligible_rows`, not from a date
    filter written here: the rows this model is finally trained on have to be
    the rows it was scored on, and the scoring cuts are not just the date.

    The **windows**, though, are cut from the full `df` — the same reason
    `build_index` takes the whole panel. Context rows outside the eligible set
    are still legitimate history.
    """
    params = {**DEFAULT_PARAMS, **params}
    quantiles = tuple(quantiles)
    index = build_index(df, feature_cols=feature_cols,
                                         lookback=lookback, date_col=date_col)
    device = resolve_device(device_name)
    sizes = sizes if sizes is not None else embedding_sizes(
        idx_cols=index["idx_cols"])

    frame = walk_forward.eligible_rows(df, lookback=lookback, date_col=date_col,
                                       test_start=test_start)
    frame = frame[purging.lookahead_safe_mask(frame, test_start, date_col=date_col)]
    model_common.assert_no_nan(frame, index["feature_cols"])

    ends = window_ends(index, frame)
    _assert_no_december(index, ends, test_start=test_start)

    scaler = modeling_prep.fit_scaler(frame, index["dynamic_cols"])
    scaled = scale_values(index["values"], scaler, index["dynamic_cols"])

    fit_rows, es_rows = model_common.split_early_stopping(
        frame, tail_days=tail_days, date_col=date_col)
    _, best_epoch = fit_with_early_stopping(
        params, index,
        window_ends(index, fit_rows), _target(fit_rows, params),
        window_ends(index, es_rows), _target(es_rows, params),
        quantiles=quantiles, sizes=sizes, device=device, scaled=scaled,
        max_epochs=max_epochs, patience=patience, lookback=lookback)

    model = fit_epochs(params, index, ends, _target(frame, params),
                       epochs=best_epoch, quantiles=quantiles, sizes=sizes,
                       device=device, scaled=scaled, lookback=lookback)

    return {
        "state_dict": {key: value.cpu() for key, value
                       in model.state_dict().items()},
        "params": params,
        "feature_cols": index["feature_cols"],
        "dynamic_cols": index["dynamic_cols"],
        "idx_cols": index["idx_cols"],
        "embedding_sizes": sizes,
        "scaler": scaler,
        "log_target": params["log_target"],
        "best_epoch": int(best_epoch),
        # The whole grid in head order, not a scalar. The column order of the
        # head is unrecoverable from `state_dict` alone, so without this a
        # reloaded model returns nineteen unlabelled numbers.
        "quantiles": quantiles,
        **model_common.target_provenance(),
        "lookback": lookback,
        "n_train": int(len(frame)),
    }


def predict_bundle(bundle: dict, panel, frame) -> np.ndarray:
    """Predict with a fitted bundle, forcing the recorded column order.

    `panel` is required and not optional: an LSTM cannot predict from a row on
    its own — it needs the 28 days behind it. Rebuilding the index from
    `bundle["feature_cols"]` is what pins the column order, so a panel whose
    columns arrive in a different order produces identical predictions.
    """
    index = build_index(
        panel, feature_cols=bundle["feature_cols"], lookback=bundle["lookback"])
    device = resolve_device(bundle["params"].get("device", "cpu"))
    quantiles = tuple(bundle["quantiles"])
    model = build_model(bundle["params"], len(bundle["dynamic_cols"]),
                        bundle["embedding_sizes"], bundle["params"]["random_state"],
                        n_quantiles=len(quantiles))
    model.load_state_dict(bundle["state_dict"])
    model.to(device)

    scaled = scale_values(index["values"], bundle["scaler"], bundle["dynamic_cols"])
    ends = window_ends(index, frame)
    prediction = np.asarray(
        predict(model, scaled, index["cats"], ends, device=device,
                lookback=bundle["lookback"]),
        dtype="float64",
    ).reshape(len(ends), len(quantiles))
    if bundle["log_target"]:
        prediction = modeling_prep.inverse_log_target(prediction)
    return np.clip(prediction, 0.0, None)


def save_bundle(bundle: dict, path: str = MODEL_FILE) -> None:
    model_common.save_bundle(bundle, path)


def load_bundle(path: str = MODEL_FILE) -> dict:
    return model_common.load_bundle(path)


def save_best_params(params: dict, path: str = BEST_PARAMS_FILE) -> None:
    model_common.save_best_params(params, path)

In [9]:
SEARCH_FOLDS = (3, 5)


select_best = model_common.select_best


def sample_search_space(
    n_candidates: int,
    seed: int = 42,
    space: Optional[dict] = None,
) -> list:
    """Distinct parameter sets drawn at random from SEARCH_SPACE.

    No affordability screen: unlike the quantile forest there is no
    leaf-storage bound to screen against — a batch of 2048 windows is 11 MB
    whatever the hidden size.

    `n_candidates` has no default on purpose. It comes from
    `candidate_budget()` and its measured inputs, so hard-coding one here
    would invite skipping the measurement.
    """
    return model_common.sample_search_space(
        space=SEARCH_SPACE if space is None else space,
        defaults=DEFAULT_PARAMS,
        n_candidates=n_candidates,
        seed=seed,
        screen=None,
    )


def run_search(
    df,
    candidates: list,
    folds: tuple = SEARCH_FOLDS,
    quantiles: tuple = QUANTILES,
    model_name: str = "lstm",
    feature_cols: Optional[list] = None,
    verbose: bool = True,
    checkpoint_path: Optional[str] = None,
    resume: bool = True,
    only: Optional[Iterable[int]] = None,
    provenance: Optional[dict] = None,
    device_name: str = "cpu",
    lookback: int = modeling_prep.LOOKBACK,
    sizes: Optional[list] = None,
):
    """Score every LSTM candidate on the search folds.

    `df` is the panel, passed to both `run_search` (which cuts eligible rows
    from it) and `bind_panel` (which cuts windows from it) — the same frame in
    both places, so the rows scored and the rows windowed cannot drift apart.
    """
    return model_common.run_search(
        df,
        candidates,
        make_fit_predict=bind_panel(df, feature_cols=feature_cols,
                                    lookback=lookback, device_name=device_name,
                                    sizes=sizes),
        search_space=SEARCH_SPACE,
        folds=folds,
        quantiles=quantiles,
        model_name=model_name,
        feature_cols=feature_cols,
        verbose=verbose,
        checkpoint_path=checkpoint_path,
        resume=resume,
        only=only,
        provenance=provenance,
    )


def run_seed_repeats(
    df,
    params: dict,
    seeds: tuple = SEED_REPEATS,
    folds: tuple = SEARCH_FOLDS,
    quantiles: tuple = QUANTILES,
    model_name: str = "lstm",
    feature_cols: Optional[list] = None,
    lookback: int = modeling_prep.LOOKBACK,
    device_name: str = "cpu",
    sizes: Optional[list] = None,
    tail_days: int = ES_TAIL_DAYS,
    max_epochs: int = MAX_EPOCHS,
    patience: int = EARLY_STOPPING_EPOCHS,
    verbose: bool = True,
    output_path: Optional[str] = None,
):
    """Re-run one configuration across seeds, on the search folds.

    The LSTM is the only model here whose weights start random, so a small K1
    gap between it and a tree model has never been separable from seed noise —
    the spread across folds mixes seed variance with data variance and cannot
    be asked to answer this. Running the *winner* at three seeds on the same
    two folds measures it directly.

    Only the winner. Repeating all thirty candidates would pay three times the
    search bill to answer a question that only matters for the configuration
    actually being reported.

    The row layout is `summarise_candidate()`'s, deliberately: the seed-42 row
    must be comparable column for column with that candidate's row in
    `lstm_search_results.csv`, and any difference between them is
    nondeterminism rather than seed variance.
    """
    frame = walk_forward.eligible_rows(df, lookback=lookback)
    make = bind_panel(df, feature_cols=feature_cols, lookback=lookback,
                      device_name=device_name, sizes=sizes, tail_days=tail_days,
                      max_epochs=max_epochs, patience=patience)

    rows = []
    for seed in seeds:
        started = time.perf_counter()
        fit_predict = make({**params, "random_state": seed},
                           quantiles=quantiles)
        results = pd.concat(
            [walk_forward.run_fold(frame, fold_id, fit_predict,
                                   model_name=model_name, quantiles=quantiles,
                                   prepared=True)
             for fold_id in folds],
            ignore_index=True,
        )
        record = {
            "seed": seed,
            **model_common.summarise_candidate(results, model_name, folds,
                                               quantiles),
            "best_epoch": model_common.reported_capacity(fit_predict),
            "elapsed_seconds": round(time.perf_counter() - started, 1),
        }
        rows.append(record)
        if verbose:
            print(f"seed {seed}: K1={record['pinball']:.4f} "
                  f"epoch={record['best_epoch'] or '-'} "
                  f"{record['elapsed_seconds']:.0f}s", flush=True)

    table = pd.DataFrame(rows)
    if output_path is not None:
        Path(output_path).parent.mkdir(parents=True, exist_ok=True)
        table.to_csv(output_path, index=False)
    return table


def seed_spread(repeats) -> dict:
    """min / mean / max / range of K1 across the seeds.

    `range` is the number that decides how the K1 table may be read: if the
    spread across seeds of one configuration exceeds the K1 distance between
    the LSTM and a tree model, that distance is not a difference between
    models and the results document has to say so (spec §2.3).
    """
    values = pd.Series(repeats["pinball"], dtype="float64").dropna()
    if values.empty:
        return {"min": float("nan"), "mean": float("nan"),
                "max": float("nan"), "range": float("nan")}
    return {
        "min": float(values.min()),
        "mean": float(values.mean()),
        "max": float(values.max()),
        "range": float(values.max() - values.min()),
    }

## 5. Setelan run & data model-ready

In [10]:
# DEVICE tidak diset di sini: notebook ini **mengukur** cpu dan cuda di sel
# benchmark, lalu memakai yang tercepat (sel "Anggaran pencarian").
# FORECAST_DEVICE boleh menimpanya — dua shard di satu sesi T4x2 harus bisa
# dipin ke cuda:0 dan cuda:1, dan benchmark tidak tahu apa-apa soal shard —
# tetapi benchmark tetap dijalankan dan angkanya tetap dicetak, supaya
# penimpaan yang ternyata lebih lambat terbaca di output.
#
# Tanpa satu pun env var, notebook ini berperilaku persis seperti sebelum
# jalur cloud ada. Rencana mesin per tahap ada di
# docs/superpowers/specs/2026-08-24-distributed-gpu-training-design.md.
SHARD = run_config.shard()
SEARCH_FILE = run_config.search_checkpoint("lstm")
RESULTS_FILE = run_config.checkpoint_path("lstm_walk_forward_results.csv")

df = pd.read_parquet(run_config.model_input_path())
print(df.shape, torch.__version__, torch.backends.mps.is_available())
print(f"QUANTILE_SET: {len(QUANTILES)} titik, "
      f"{QUANTILES[0]}..{QUANTILES[-1]}")

# Path artefak datang dari run_config, bukan dari konstanta modul, supaya FORECAST_CHECKPOINT_DIR
# berlaku untuk seluruh sel — termasuk §11 dan §12 yang membaca ulang tabel hasil. Tanpa env var
# apa pun keduanya menunjuk berkas yang sama.
print(f"SEARCH_FILE  = {SEARCH_FILE}")
print(f"RESULTS_FILE = {RESULTS_FILE}")

(1502522, 82) 2.8.0 True
QUANTILE_SET: 19 titik, 0.05..0.95
SEARCH_FILE  = /Users/ramapdp/Project/Personal/forecast-scm/dataset/model_ready/lstm_search_results.csv
RESULTS_FILE = /Users/ramapdp/Project/Personal/forecast-scm/dataset/model_ready/lstm_walk_forward_results.csv


## 6. Benchmark

In [11]:
# Satu putaran dua-fit di fold 5 dengan DEFAULT_PARAMS, di CPU. MPS tidak diukur ujung-ke-ujung:
# probe 15 batch di fold 5 (2026-08-19) mencatat 0,392 s/batch di MPS lawan 0,193 s/batch di CPU
# — tidak ada kernel LSTM ter-fusi di MPS pada hidden size ini, jadi menjalankan benchmark penuh
# di sana hanya menghabiskan ~4 jam untuk mengonfirmasi perangkat yang sudah kalah 2x.
#
# Yang diukur: detik per epoch, epoch tempat early stopping mendarat, dan peak RSS. Ketiganya
# yang mengisi rumus anggaran di §2.2 spec.
import time

frame = walk_forward.eligible_rows(df)
split = walk_forward.prepare_fold(frame, 5, prepared=True)
print('train', len(split['train']), 'valid', len(split['valid']))

benchmark = {}
# Setiap device diukur, yang tidak ada di mesin ini dilewati.
# resolve_device() menolak 'cuda' dengan ValueError kalau tidak
# tersedia, dan bind_panel() memeriksanya sebelum membangun indeks
# jendela — jadi device yang mustahil tidak menghabiskan sort 1,5 juta
# baris lebih dulu. DEVICE terpilih di sel berikutnya adalah yang
# sec_per_epoch-nya terkecil, diukur, bukan diasumsikan.
for device_name in ('cpu', 'cuda'):
    try:
        make = bind_panel(df, device_name=device_name)
    except ValueError as failure:
        print(device_name, 'dilewati:', failure)
        continue
    fit_predict = make(DEFAULT_PARAMS, quantiles=QUANTILES)
    started = time.time()
    prediction = fit_predict(split['train'], split['valid'])
    elapsed = time.time() - started
    best_epoch = fit_predict.best_epochs[0]
    # elapsed covers both fits: best_epoch + patience epochs in the
    # first, best_epoch in the second.
    epochs_run = 2 * best_epoch + EARLY_STOPPING_EPOCHS
    headline = min(range(len(QUANTILES)),
                   key=lambda i: abs(QUANTILES[i] - evaluation.DEFAULT_ALPHA))
    peak_bytes = model_common.peak_rss_bytes()  # None di Windows
    benchmark[device_name] = {
        'wall_seconds': elapsed,
        'best_epoch': best_epoch,
        'sec_per_epoch': elapsed / epochs_run,
        'peak_rss_gb': None if peak_bytes is None else peak_bytes / 1e9,
        'pred_mean_q90': float(prediction[:, headline].mean()),
        'pred_max_q90': float(prediction[:, headline].max()),
        # Head komposit tidak menjamin monotonisitas. Diukur, bukan diasumsikan.
        'crossing_rate': float(evaluation.crossing_rate(prediction, QUANTILES)),
    }
    print(device_name, benchmark[device_name])

pd.DataFrame(benchmark).T

train 1292778 valid 59629


cpu {'wall_seconds': 961.123242855072, 'best_epoch': 2, 'sec_per_epoch': 106.79147142834134, 'peak_rss_gb': 5.928779776, 'pred_mean_q90': 43.75163627359618, 'pred_max_q90': 1622.4483642578125, 'crossing_rate': 0.17845343708598166}
cuda dilewati: CUDA tidak tersedia di mesin ini


,wall_seconds,best_epoch,sec_per_epoch,peak_rss_gb,pred_mean_q90,pred_max_q90,crossing_rate
cpu,961.123243,2.0,106.791471,5.92878,43.751636,1622.448364,0.178453


## 7. Anggaran pencarian

In [12]:
# N datang dari angka benchmark, bukan dari tebakan. Kalau rumusnya jatuh di bawah 6,
# candidate_budget melempar ValueError — itu sinyal untuk memperkecil ruang search, bukan
# menaikkan plafon 8 jam.
# Yang diukur tetap yang dilaporkan: MEASURED_DEVICE adalah pemenang
# benchmark, DEVICE adalah yang benar-benar dipakai. Keduanya dicetak, jadi
# penimpaan lewat FORECAST_DEVICE yang ternyata lebih lambat terlihat di
# output alih-alih hilang diam-diam.
MEASURED_DEVICE = min(benchmark, key=lambda name: benchmark[name]['sec_per_epoch'])
DEVICE = run_config.device(MEASURED_DEVICE)
measured = benchmark[MEASURED_DEVICE]

# N **dipatok** 30, setara XGBoost (spec §2.2, penyetaraan anggaran
# 2026-08-24). candidate_budget() tetap dijalankan, tetapi sebagai
# *pengukuran ongkos*, bukan sebagai penentu N: plafon 8 jam sudah
# ditinggalkan secara sadar, dan wall clock sebenarnya dicatat di
# docs/hasil-modeling-md sebagai ongkos terukur.
try:
    implied = candidate_budget(
        sec_per_epoch=measured['sec_per_epoch'],
        best_epoch=measured['best_epoch'],
    )
except ValueError as failure:
    implied = f'<{MIN_CANDIDATES} ({failure})'

N_CANDIDATES = N_CANDIDATES
print('device tercepat  :', MEASURED_DEVICE, '(diukur)')
print('device dipakai   :', DEVICE,
      '<- ditimpa FORECAST_DEVICE' if DEVICE != MEASURED_DEVICE else '')
print(run_config.describe(DEVICE))
print('sec_per_epoch    :', round(measured['sec_per_epoch'], 1))
print('best_epoch       :', measured['best_epoch'])
print('N menurut plafon :', implied, '(informatif saja)')
print('N yang dipakai   :', N_CANDIDATES, '- dipatok, setara XGBoost')
print('ruang pencarian  :', np.prod([len(v) for v in SEARCH_SPACE.values()]),
      'titik')

device tercepat  : cpu (diukur)
device dipakai   : cpu 
device: cpu | seluruh kandidat | input: /Users/ramapdp/Project/Personal/forecast-scm/dataset/model_ready/model_input.parquet | checkpoint: /Users/ramapdp/Project/Personal/forecast-scm/dataset/model_ready
sec_per_epoch    : 106.8
best_epoch       : 2
N menurut plafon : 14 (informatif saja)
N yang dipakai   : 30 - dipatok, setara XGBoost
ruang pencarian  : 144 titik


## 8. Pencarian hyperparameter

In [13]:
# Fold 3 dan 5 saja, seed 42, kriteria pinball@0,9 gabungan berbobot jumlah baris. Checkpoint
# di-flush tiap kandidat selesai.
# Ke-30 kandidat dijalankan ulang di bawah head multi-kuantil, pada ruang
# 144 yang sudah dipulihkan (num_layers dan hidden_size kembali). Artefak run
# kuantil-tunggal sudah diganti nama menjadi `*.single-quantile.bak.*`
# (2026-08-24), jadi sel ini mulai dari nol; kalau berkas tanpa kolom
# `headline_quantile` muncul lagi di jalur checkpoint, guard checkpoint
# menolaknya.
#
# Dengan FORECAST_SHARD diset, mesin ini hanya menjalankan bagiannya, dengan
# penomoran yang tetap absolut terhadap seed 42 — itulah yang membuat dua
# shard bisa disatukan `model_common.merge_shards()` nanti.
candidates = sample_search_space(N_CANDIDATES, seed=42)
search_results = run_search(
    df, candidates,
    checkpoint_path=SEARCH_FILE,
    device_name=DEVICE,
    only=SHARD,
    provenance=run_config.provenance(DEVICE),
)
# `pinball` di sini adalah K1. Kolom *_headline dibaca di tau=0,9.
search_results.sort_values('pinball').head(10)

melanjutkan dari checkpoint: 30 kandidat sudah selesai


,candidate_id,batch_size,dropout,hidden_size,learning_rate,log_target,num_layers,device,commit,pinball,mae_headline,coverage_headline,fill_rate_headline,coverage_gap,crossing_rate,headline_quantile,best_epoch,elapsed_seconds,error
21,21,2048,0.0,256,0.0003,True,2,cuda,e074421,2.861673,14.443382,0.894238,0.957571,0.143938,0.442174,0.9,"5,9",939.7,NaN
12,12,2048,0.0,64,0.0003,True,2,cuda,e074421,2.896286,14.746972,0.898442,0.958278,0.143222,0.474718,0.9,"13,17",691.9,NaN
5,5,1024,0.0,128,0.0003,True,2,cuda,e074421,2.915091,15.526984,0.912212,0.962437,0.147433,0.422425,0.9,"3,9",590.4,NaN
13,13,2048,0.0,64,0.0010,False,2,cuda,e074421,2.924167,13.563376,0.886638,0.952430,0.137375,0.379776,0.9,"7,8",404.7,NaN
23,23,1024,0.0,256,0.0010,False,2,cuda,e074421,2.926138,13.539706,0.889074,0.953164,0.138736,0.332286,0.9,"3,3",499.8,NaN
29,29,2048,0.2,128,0.0003,False,1,cuda,e074421,2.935906,14.722273,0.890242,0.957405,0.140210,0.218094,0.9,"13,13",640.7,NaN
6,6,2048,0.0,256,0.0010,False,2,cuda,e074421,2.937206,14.178238,0.918882,0.954806,0.161181,0.251045,0.9,"4,2",565.5,NaN
17,17,1024,0.0,256,0.0003,True,1,cuda,e074421,2.939418,16.499489,0.911790,0.964523,0.155928,0.442520,0.9,"5,9",637.2,NaN
24,24,2048,0.2,256,0.0010,False,1,cuda,e074421,2.940648,14.202924,0.900685,0.955413,0.148560,0.187978,0.9,"3,2",250.4,NaN
25,25,1024,0.0,128,0.0003,False,1,cuda,e074421,2.941699,14.609679,0.909984,0.957066,0.150417,0.250330,0.9,"7,6",411.5,NaN


## 9. Walk-forward final

In [14]:
# Pemenang dipilih di sini dan tidak di mana pun lagi.
#
# Terpisah dari walk-forward di sel berikutnya karena keduanya berjalan di
# mesin yang berbeda (keputusan 2026-08-26): pencarian dan pengulangan tiga
# seed di PC GPU, walk-forward dan fit final di Mac. Sel ini murah di kedua
# mesin — dengan checkpoint penuh, sel pencarian di atas hanya membaca CSV —
# jadi `best` tersedia di PC untuk sel pengulangan seed tanpa membayar
# walk-forward, dan tersedia lagi di Mac tanpa membayar pencarian.
#
# Berhenti di sini kalau ini run bershard: memilih pemenang dari sebagian
# kandidat menghasilkan angka yang tampak sepenuhnya wajar. Gabungkan seluruh
# shard dengan model_common.merge_shards() di satu mesin lebih dulu, lalu
# jalankan sel ini di sana tanpa FORECAST_SHARD.
assert SHARD is None, (
    "run bershard: jangan pilih pemenang dari sebagian kandidat — "
    "gabungkan seluruh shard dengan model_common.merge_shards() lebih dulu"
)

best = select_best(search_results, candidates)
save_best_params(best)
print(best)


{'hidden_size': 256, 'num_layers': 2, 'dropout': 0.0, 'learning_rate': 0.0003, 'batch_size': 2048, 'log_target': True, 'grad_clip': 1.0, 'random_state': 42}


In [15]:
# Pemenang dijalankan ulang di kelima fold. Wajib di mesin yang sama dengan
# walk-forward kedua model lain — wall time-nya masuk K3, dan K3-lah yang
# menentukan pemenang saat K1 seri. run_walk_forward() juga tidak punya
# checkpoint: sesi yang terpotong di tengah kehilangan seluruhnya.
make = bind_panel(df, device_name=DEVICE)
fit_predict = make(best, quantiles=QUANTILES)
results = walk_forward.run_walk_forward(
    df, fit_predict, model_name='lstm', quantiles=QUANTILES)
results.to_csv(RESULTS_FILE, index=False)
print('best_epoch per fold:', fit_predict.best_epochs)
print(f"K1: {walk_forward.pooled_k1(results, 'lstm'):.4f}")


best_epoch per fold: [6, 11, 5, 8, 9]
K1: 2.8828


## 10. Pengulangan tiga seed pada pemenang

In [16]:
# LSTM satu-satunya model di perbandingan ini yang inisialisasi bobotnya acak, jadi selisih K1
# kecil antara ia dan model pohon selama ini tidak dapat dipisahkan dari derau seed. Konfigurasi
# pemenang — dan hanya itu — diulang pada seed 42/43/44 di fold pencarian yang sama (3 dan 5).
#
# Baris seed 42 harus sama persis dengan baris pemenang di lstm_search_results.csv. Kalau tidak,
# yang terukur bukan varians seed melainkan nondeterminisme yang belum tertangkap — dan itu
# temuan tersendiri, bukan pembulatan yang boleh dilewatkan.
#
# Dijalankan di mesin yang sama dengan pencariannya (PC GPU sejak 2026-08-26),
# bukan di Mac. Pemeriksaan konsistensi di bawah membandingkan baris seed 42
# dengan baris pemenang di lstm_search_results.csv, dan dua device berselisih
# setingkat paritas (0,124%, Bagian 3bis spec GPU) — selisih yang akan terbaca
# sebagai nondeterminisme padahal ia cuma hardware. Di Mac sel ini karena itu
# membaca kembali CSV yang dibawa pulang alih-alih menghitung ulang: angkanya
# tetap angka PC, dan `spread` tersedia untuk §13 tanpa membayar ~7 jam kedua
# kalinya. run_seed_repeats() sendiri tidak punya resume dan menimpa
# keluarannya, jadi penjagaannya harus di sini.
sudah = (pd.read_csv(SEED_REPEATS_FILE) if Path(SEED_REPEATS_FILE).exists()
         else None)
if sudah is not None and sorted(int(s) for s in sudah['seed']) == sorted(SEED_REPEATS):
    print(f"membaca kembali {SEED_REPEATS_FILE} — {len(sudah)} seed sudah dijalankan, "
          f"tidak diulang")
    repeats = sudah
else:
    repeats = run_seed_repeats(
        df, best, seeds=SEED_REPEATS, folds=SEARCH_FOLDS,
        device_name=DEVICE, output_path=SEED_REPEATS_FILE)

spread = seed_spread(repeats)
print(f"K1 antar seed: min {spread['min']:.4f}  mean {spread['mean']:.4f}  "
      f"max {spread['max']:.4f}  rentang {spread['range']:.4f}")

# Konsistensi terhadap pencarian: baris seed 42 vs baris pemenang.
winner = search_results.loc[search_results['pinball'].idxmin()]
delta = float(repeats.loc[repeats['seed'] == 42, 'pinball'].iloc[0]) - float(winner['pinball'])
print(f"selisih seed-42 terhadap baris pemenang di search: {delta:+.6f} "
      f"({'konsisten' if abs(delta) < 1e-6 else 'PERIKSA - ada nondeterminisme'})")

repeats

membaca kembali /Users/ramapdp/Project/Personal/forecast-scm/dataset/model_ready/lstm_seed_repeats.csv — 3 seed sudah dijalankan, tidak diulang
K1 antar seed: min 2.8399  mean 2.9310  max 3.0915  rentang 0.2517
selisih seed-42 terhadap baris pemenang di search: +0.000000 (konsisten)


,seed,pinball,mae_headline,coverage_headline,fill_rate_headline,coverage_gap,crossing_rate,headline_quantile,best_epoch,elapsed_seconds
0,42,2.861673,14.443382,0.894238,0.957571,0.143938,0.442174,0.9,"5,9",917.0
1,43,3.091536,16.488114,0.908585,0.964095,0.158333,0.462423,0.9,"6,12",1112.6
2,44,2.839871,13.872771,0.896398,0.956209,0.138288,0.478937,0.9,"7,10",1062.2


## 11. Model final

In [17]:
bundle = fit_final(df, best, device_name=DEVICE)
save_bundle(bundle)
print(f"best_epoch {bundle['best_epoch']}, n_train {bundle['n_train']:,}, "
      f"{len(bundle['quantiles'])} titik kuantil "
      f"({bundle['quantiles'][0]}..{bundle['quantiles'][-1]})")

best_epoch 5, n_train 1,349,011, 19 titik kuantil (0.05..0.95)


## 12. Hasil

In [18]:
results = pd.read_csv(RESULTS_FILE)
HEADLINE = evaluation.DEFAULT_ALPHA

print("=== K1 (rata-rata pinball lintas QUANTILE_SET, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} K1 {walk_forward.pooled_k1(results, model):7.4f}")

print("\n=== per fold, K1 ===")
print(results[results["group_col"].isna()]
      .pivot_table(index="model", columns="fold_id", values="pinball").round(3))

print(f"\n=== per fold, pinball di tau={HEADLINE} (angka headline B-9) ===")
headline_rows = results[results["group_col"].isna()
                        & ((results["quantile"] - HEADLINE).abs() < 1e-9)]
print(headline_rows.pivot_table(index="model", columns="fold_id",
                                values="pinball").round(3))

for group_col in walk_forward.GROUP_COLS:
    print(f"\n=== per {group_col} (K1, pooled over folds) ===")
    grouped = results[results["group_col"] == group_col]
    # Kolom dipilih sebelum apply: tanpa itu pandas ikut menyertakan kolom
    # pengelompokan dan mengeluarkan FutureWarning di setiap sel.
    table = (grouped.assign(weighted=grouped["pinball"] * grouped["n"])
                    .groupby(["model", "group_value"], observed=True)[["weighted", "n"]]
                    .apply(lambda part: part["weighted"].sum() / part["n"].sum())
                    .unstack())
    print(table.round(3))

# pooled_metric menolak dirata-ratakan lintas kuantil untuk metrik selain
# pinball/crossing_rate — coverage di 0,05 dan di 0,95 menjawab pertanyaan
# yang berbeda. Jadi ketiganya dibaca di tau headline, eksplisit.
print(f"\n=== coverage / fill rate di tau={HEADLINE} (overall, pooled) ===")
for model in results["model"].unique():
    print(f"{model:20s} "
          f"coverage {walk_forward.pooled_metric(results, model, 'coverage', quantile=HEADLINE):6.3f}  "
          f"fill_rate {walk_forward.pooled_metric(results, model, 'fill_rate', quantile=HEADLINE):6.3f}  "
          f"shortfall {walk_forward.pooled_metric(results, model, 'shortfall_units', quantile=HEADLINE):9.1f}  "
          f"crossing {walk_forward.pooled_metric(results, model, 'crossing_rate'):6.4f}")

print("\n=== K2: coverage per titik kuantil (lstm) ===")
print(walk_forward.coverage_by_quantile(results, "lstm").round(4)
      .to_string(index=False))

=== K1 (rata-rata pinball lintas QUANTILE_SET, pooled) ===
lstm                 K1  2.8828
naive_zero           K1 14.7469
naive_lag_1          K1  8.1755
naive_roll_mean_7    K1  4.8231

=== per fold, K1 ===
fold_id                 1       2       3       4       5
model                                                    
lstm                2.706   2.874   2.721   3.079   3.078
naive_lag_1         7.955   8.533   8.108   7.971   8.307
naive_roll_mean_7   4.664   4.981   4.668   4.937   4.872
naive_zero         14.696  15.141  13.249  14.566  16.289

=== per fold, pinball di tau=0.9 (angka headline B-9) ===
fold_id                 1       2       3       4       5
model                                                    
lstm                2.287   2.420   2.316   2.745   2.574
naive_lag_1         8.355   8.469   8.045   8.526   8.372
naive_roll_mean_7   4.249   4.566   4.034   4.783   4.970
naive_zero         26.453  27.254  23.849  26.219  29.320

=== per demand_segment (K1, pooled 

## 13. Head-to-head tiga arah

In [19]:
# Sah karena ketiganya dinilai di baris identik — dijamin walk_forward.eligible_rows(). Potongan
# kedua (fold 1, 2, 4) adalah angka bersihnya: tidak ada model yang memakai fold itu untuk
# seleksi.

HEADLINE = evaluation.DEFAULT_ALPHA
tables = {
    'lstm': pd.read_csv(RESULTS_FILE),
    'xgboost': pd.read_csv(run_config.checkpoint_path("xgb_walk_forward_results.csv")),
    'random_forest': pd.read_csv(run_config.checkpoint_path("rf_walk_forward_results.csv")),
}
rows = []
for name, table in tables.items():
    rows.append({
        'model': name,
        'k1_5_folds': walk_forward.pooled_k1(table, name),
        'k1_folds_124': walk_forward.pooled_k1(table, name, folds=(1, 2, 4)),
        'mae_q90': walk_forward.pooled_metric(
            table, name, metric='mae', quantile=HEADLINE),
        'coverage_q90': walk_forward.pooled_metric(
            table, name, metric='coverage', quantile=HEADLINE),
        'crossing_rate': walk_forward.pooled_metric(
            table, name, metric='crossing_rate'),
    })
table = pd.DataFrame(rows).sort_values('k1_folds_124')

# Jarak K1 antar model dibaca bersama rentang antar seed di atas: kalau
# rentang seed LSTM melebihi jarak ini, jarak itu bukan perbedaan antar model.
print(f"rentang K1 antar seed (LSTM): {spread['range']:.4f}")
print(f"jarak K1 terkecil antar model: "
      f"{table['k1_folds_124'].diff().abs().min():.4f}")
table

rentang K1 antar seed (LSTM): 0.2517
jarak K1 terkecil antar model: 0.0310


,model,k1_5_folds,k1_folds_124,mae_q90,coverage_q90,crossing_rate
2,random_forest,2.862088,2.850806,15.081272,0.928126,0.000000
0,lstm,2.882785,2.881832,13.929030,0.906166,0.434473
1,xgboost,2.919705,2.943314,13.408394,0.902204,0.976738


## 14. *(Opsional)* Cek sinkron dengan `utils/`

In [20]:
import inspect
import re

from utils.modelling import model_lstm as _ref_lstm
from utils.modelling import sequence_windows as _ref_sw

_QUALIFIER = re.compile(r"\bsequence_windows\.")
# SEARCH_FILE dan RESULTS_FILE sengaja datang dari run_config di §5, jadi nilainya boleh berbeda
# dari konstanta modul begitu FORECAST_CHECKPOINT_DIR diset. N_CANDIDATES ditimpa di §7.
_LEWATI = {"find_base_dir", "SEARCH_FILE", "RESULTS_FILE", "N_CANDIDATES", "DEVICE",
           "_QUALIFIER", "_LEWATI", "_ref_lstm", "_ref_sw", "_beda", "_n_fungsi",
           "_n_konstanta", "_kode", "_kode_objek"}


def _kode(fn) -> str:
    """Kode sumber tanpa qualifier sequence_windows — modul itu kini satu namespace di sini."""
    return _QUALIFIER.sub("", inspect.getsource(fn))


def _kode_objek(obj) -> str:
    """Sumber sebuah objek; kelas dibandingkan lewat method-nya.

    inspect.getsource() pada sebuah kelas mencari definisinya di berkas sumber, dan kelas yang
    lahir di sel notebook tidak punya berkas. Method-nya punya code object, jadi jalurnya ada.
    """
    if inspect.isclass(obj):
        return "\n".join(_kode(f) for n, f in sorted(vars(obj).items()) if inspect.isfunction(f))
    return _kode(obj)


_beda, _n_fungsi, _n_konstanta = [], 0, 0
for nama, obj in sorted(globals().items()):
    if nama in _LEWATI or nama.startswith("__"):
        continue
    ref = getattr(_ref_lstm, nama, None) or getattr(_ref_sw, nama, None)
    if ref is None:
        continue
    if inspect.isfunction(obj) or inspect.isclass(obj):
        _n_fungsi += 1
        if _kode_objek(obj) != _kode_objek(ref):
            _beda.append(nama)
    elif nama.isupper():
        _n_konstanta += 1
        if obj != ref:
            _beda.append(nama)

if _beda:
    print("BERBEDA dari utils/ — salin ulang atau samakan: " + ", ".join(_beda))
else:
    print(f"Sinkron: {_n_fungsi} fungsi/kelas + {_n_konstanta} konstanta identik dengan "
          f"utils/modelling/model_lstm.py + sequence_windows.py")

Sinkron: 36 fungsi/kelas + 16 konstanta identik dengan utils/modelling/model_lstm.py + sequence_windows.py
